In [1]:
### ---- START M.L. Specific Imports --- ###
from xml.parsers.expat import model
from assignment1_data.assignment1_data.mnist_dataloader import create_dataloaders
import torch
import torch.nn as nn
import tensorflow as tf
from torch.optim import SGD
from torchvision import transforms
from torchvision.transforms import v2
### ---- END M.L. Specific Imports --- ###

### ---- START General Imports --- ###
import matplotlib.pyplot as PLT
import numpy as np
import os
import csv
import pandas as pd
### ---- End General Imports --- ###

class Fit_Predict():
    def __init__(self, model, optimizer, loss, 
                 batch_size, data_loc, 
                 model_label=None):
        """
        Def:
        --- Constructor Class argumnets ---
        :parma ... model:
        :param ... optimizer:...
        """
        self.model = model
        self.optim = optimizer
        self.loss = loss
        self.model_label:str = model_label

        # 1. Data loading...
        self.train_loader, self.test_loader, self.val_loader = self.pipeline(batch_size, data_loc)

    def train(self, epochs):
        """
        Def:
        :param
        :param
        """

        # 1. Creating the required directory hierarchy
        self._utils_Directory_Exec()

        # 2. Equipping M.L. utils (e.g., define criterion and optimizer)
        # 2.1 Creating history-holder containers (e.g., "model-name"; "model-name"_accuracy_container)
        loss_container_train, loss_container_te = [], []

        for epoch in range(epochs):
            _loss_per_epoch_tr:float = 0.0
            _loss_per_epoch_te:float = 0.0
            #TODO Optimize the looping mechanism for faster iteration;

            # 3. Training kicks off
            self.model.train()
            for cl, ns, label in self.train_loader:
                self.optim.zero_grad()
                output = self.model(ns)
                # 3.1 Quantifying the quality of the output; Backpropagate; Optimizer step
                _loss_tr = self.loss(output,cl) # Compare the batch of recon against the batch with clear
                _loss_tr.backward()
                self.optim.step()
                # 3.2 Keep track of performance
                _loss_per_epoch_tr+=_loss_tr.item() #TODO is .item() extraneous?

            loss_container_train.append(_loss_per_epoch_tr/len(self.train_loader))
            # 4. Testing kicks off
            self.model.eval()
            with torch.no_grad():
                for cl, ns, label in self.test_loader:
                    output = self.model(ns)
                    _loss_te = self.loss(output, cl)
                    _loss_per_epoch_te+=_loss_te.item()
                loss_container_te.append(_loss_per_epoch_te/len(self.test_loader))
            print("Epoch {} yielded: Train Loss {:.2f} | Test Loss {:.2f} ".format(epoch, loss_container_train[epoch], loss_container_te[epoch]))
            # 5. Ensures the weights are saved one epoch at a time;
            torch.save(self.model.state_dict(), f'weights\{self.model_label}\{self.model_label}model_weights_epoch{epoch}.pth')
        # 6. Ensures both containers are being saved under `high-capacity_accuracy_container\history.csv`
        # 6.1 Convert the lists to a pandas dataframe for easier saving
        raw_history_frame = pd.DataFrame({
            "train_loss": loss_container_train, 
            "test_loss": loss_container_te
        })
        # 6.2 Save the dataframe as a .csv file
        raw_history_frame.to_csv(f"weights\{self.model_label}_accuracy_container\history.csv")

            
    def pipeline(self, batch_size, data_loc):
        """
        Def:
        :param...
        :param...
        """

        def __flattener(data:tuple):
            f = nn.Flatten(start_dim=0)
            flattened = [f(item) if isinstance(item, torch.Tensor) else item for item in data[:-1]]
            return tuple(flattened) + (data[-1],)
    
        basisc_transf = v2.Compose([
            # transforms.ToTensor(), <- Data is of tensor nature by definition
            # v2.Normalize(mean=[0.5],std=[0.5]),
            __flattener
        ])
        
        train, test, val = create_dataloaders(data_loc=data_loc, batch_size=batch_size, transform=basisc_transf)
        
        return train, test, val

    @staticmethod
    def report(history:list|str, off=True):
        """
        Motivation:
        Def:

        :param list|str history: It could either hold the path leading to the .csv that holds the lists of train & test losses 
        or directly include the list (e.g. a lists of lists or a mere list);
        :param
        """
        if off:
            pass
        elif isinstance(history, list) == True or isinstance(history, str):
            pass
        else:
            pass
      
            
    
    def _utils_Directory_Exec(self, execution:bool=True):
        if execution:
            # 1. Create and check if directories wherein history is stored exist
            # 1.1 For weights amd containers that hold MSE-loss scores
            if self.model_label != None and ('weights' in os.listdir()) == False:
                os.makedirs('weights')
                if (self.model_label in os.listdir('weights')) == False:
                    os.makedirs(f'weights\{self.model_label}')
                    os.makedirs(f'weights\{self.model_label}_accuracy_container')
        else:
            pass

In [2]:
# TODO Document all of the function bellow if time allows;

"""
The current models holds ReLu as non-linear activation units.
The network should be a class and its constructor `__init__` should accept an argument defining the width of the network (e.g., layer sizes) - Done

e.g., YourClassName([32**2, 24**2, 16**2, 32**2]) - Done
- 32^2 inputs (one image at a time); - Done
- two hidden layers 24^2 and 16^2; - Done
- 32^2 outputs(the reconstruction of the image) - Done
"""

class BasicFCN(nn.Module):
    def __init__(self, width:list, activation:list):
        super().__init__() # Calling in the constructor of the inhereted class `nn.Module`
        self.width = width
        hid_actv_function = getattr(nn, activation[0])
        final_actv_function = getattr(nn, activation[1])

        # ...start building the network within the constructor 
        self.fcn_input = nn.Linear(in_features=width[0], out_features=width[1], bias=True)
        self.btcnorm1 = nn.BatchNorm1d(width[1])
        self.activation1 = hid_actv_function(0.01)
        self.fcn_hidlay1 = nn.Linear(in_features=width[1], out_features=width[2], bias=True)
        self.btcnorm2 = nn.BatchNorm1d(width[2])
        self.activation2 = hid_actv_function(0.01)
        self.fcn_hidlay2 = nn.Linear(in_features=width[2], out_features=width[3], bias=True)
        self.btcnorm3 = nn.BatchNorm1d(width[3])
        self.activation3 = hid_actv_function(0.01)
        self.fcn_hidlay3 = nn.Linear(in_features=width[3], out_features=width[4], bias=True)
        self.activation4 = hid_actv_function(0.01)
        self.btcnorm4 = nn.BatchNorm1d(width[4])
        self.fcn_hidlay4 = nn.Linear(in_features=width[4], out_features=width[5], bias=True)
        self.btcnorm5 = nn.BatchNorm1d(width[5])
        self.activation5 = hid_actv_function(0.01) # TODO Argue the reason that made you addopt `0.01`
        self.fcn_hidlay5 = nn.Linear(in_features=width[5], out_features=width[6], bias=True)
        self.activation6 = final_actv_function()
  
    def forward(self, x):
        x = self.activation1(self.btcnorm1(self.fcn_input(x)))
        x = self.activation2(self.btcnorm2(self.fcn_hidlay1(x)))
        x = self.activation3(self.btcnorm3(self.fcn_hidlay2(x)))
        x = self.activation4(self.btcnorm4(self.fcn_hidlay3(x)))
        x = self.activation5(self.btcnorm5(self.fcn_hidlay4(x)))
    
        # The Output "Head"
        x = self.fcn_hidlay5(x) 
        x = self.activation6(x) 

        return x

In [5]:
### --- START Experimental Design Setings definition --- ###
#TODO Trasnform this eventually in a parser...
#TODO The parser will be wrapped into a different function wherefrom arguments are extracted and used in __name__=="__main__"
batch_size:int = 64
loc:str = r"/DataSets"
epochs:int = 5
width_default:list = [32**2, 24**2, 16**2, 32**2]
width_complex:list = [32**2, 24**2, 22**2, 20**2, 22**2, 24**2, 32**2]
width_less_complex:list = [32**2, 16**2, 32**2]

hidd_activation:str = "LeakyReLU"
final_activation:str = "Tanh"
activation:list = [hidd_activation, final_activation]

model = BasicFCN(activation=activation, width=width_complex)
optimizer = SGD(params=model.parameters(), lr=0.01) # optimizer defined outside of the training loop
# optimizer = AdamW(params=model.parameters(), lr=0.0005)
#TODO Re-initialize loss inside the loop in order 
loss = nn.MSELoss()
model_label:str = 'high-capacity'

In [6]:
trial_1 = Fit_Predict(model=model,loss=loss,data_loc=loc,
                      optimizer=optimizer,batch_size=batch_size,
                      model_label=model_label)

trial_1.train(epochs=epochs)

Epoch 0 yielded: Train Loss 0.54 | Test Loss 0.33 
Epoch 1 yielded: Train Loss 0.27 | Test Loss 0.24 
Epoch 2 yielded: Train Loss 0.21 | Test Loss 0.20 
Epoch 3 yielded: Train Loss 0.18 | Test Loss 0.17 
Epoch 4 yielded: Train Loss 0.16 | Test Loss 0.16 
